<a href="https://colab.research.google.com/github/rutgers-ml-ai/natural-lang-processing-spring26/blob/main/nlp_week_5_agentic_ai_frameworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# agentic ai with frameworks
by mehek niwas

april 2026

## what is crewAIIIII?

- framework for "multi-agent orchestration"
- it lets you define a crew of AI agents w/ role, goal, and a set of tools

| Concept | CrewAI |
|---|---|
| An agent | `Agent(role=..., goal=..., backstory=..., tools=...)` |
| Running an agent | `Task(description=..., agent=..., expected_output=...)` |
| Orchestration | `Crew(agents=[...], tasks=[...], process=...)` |
| Tool definition | `@tool` decorator (same idea, different import) |
(i made claude generate this lovely table)

## workflow diagram
![diagram](https://i.ibb.co/Ngh45GW5/ai-graph-1.png)

## install packages

In [24]:
!pip install crewai crewai-tools litellm requests -q

## imports

get a Mistral API key: https://docs.mistral.ai/getting-started/quickstart

here, we are using Mistral through LiteLLM (a wrapper that makes it easy to switch better models, etc)

In [25]:
import os
import json
import requests
from google.colab import userdata

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

In [26]:
try:
    os.environ['MISTRAL_API_KEY'] = userdata.get('MISTRAL_API_KEY')
except Exception:
    os.environ['MISTRAL_API_KEY'] = ""  # <-- paste key here if not using Colab secrets

## setup card deck (deckofcardsapi)

same as last week demo

In [27]:
def setup_deck():
  print("set up da card deck")
  res = requests.get("https://deckofcardsapi.com/api/deck/new/shuffle/?deck_count=1").json()
  deck_id = res['deck_id']
  draw_res = requests.get(f"https://deckofcardsapi.com/api/deck/{deck_id}/draw/?count=52").json()

  cards = []
  for c in draw_res['cards']:
    code = c['code']
    if code.startswith('0'):
        code = '10' + code[1:]
    cards.append(code)
  return cards

RANKS = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K']

## global game state

global state variables to store information during the game

In [28]:
game = None           # BSGame instance
current_player = None # Name of the agent currently acting
all_agents = {}       # name -> CrewAI Agent
chat_log = []         # Shared log of chat messages all agents can see

## tools

- tools are defined with `@tool` from `crewai.tools` (@tool is a universal thing across agentic frameworks).

- crewAI reads the **docstring** and **type hints** to auto-generate the JSON schema (last week we did this manually).

- you can just use AI to create the doc strings

In [29]:
@tool
def play_cards(cards: list) -> str:
  """Play 1 to 4 cards from your hand face-down. You must declare them as the target rank.
  You can lie about what the cards actually are — that's the whole game!

  Args:
    cards: List of card codes from your hand to play (e.g. ['AS', 'AD']).
  """
  global game, current_player

  hand = game.hands[current_player]

  if not (1 <= len(cards) <= 4):
      return "Error: You must play between 1 and 4 cards."
  if not all(c in hand for c in cards):
      return f"Error: Some cards aren't in your hand. Your hand is: {hand}"

  for c in cards:
      hand.remove(c)
  game.center_pile.extend(cards)

  target_rank = game.get_target_rank()
  game.last_play = {"player": current_player, "count": len(cards), "cards": cards}

  print(f"🃏 {current_player} played {len(cards)} card(s) claiming to be {target_rank}s.")
  return f"Cards played successfully. You claimed they were {target_rank}s."


@tool
def call_bs() -> str:
  """Call BS (Cheat) on the last player's play if you think they are lying."""
  return "BS_CALLED"   # sentinel — main loop checks for this


@tool
def pass_turn() -> str:
  """Decline to call BS. Allow the game to proceed."""
  return "PASSED"


@tool
def get_opponent_card_count(player_name: str) -> str:
  """Check how many cards a specific opponent has left in their hand.

  Args:
      player_name: Name of the player to check.
  """
  count = len(game.hands.get(player_name, []))
  return f"{player_name} has {count} cards."


@tool
def chat(message: str) -> str:
  """Chat with the table. Send a short message to all other players.

  Args:
      message: What you want to say. Plain text only, no markdown.
  """
  global current_player, chat_log
  print(f"💬 [{current_player}]: {message}")
  chat_log.append(f"{current_player}: {message}")
  return "Chat sent."


PLAYER_TOOLS = [play_cards, call_bs, pass_turn, get_opponent_card_count, chat]

## game engine

same as last week's demo


In [30]:
class BSGame:
  def __init__(self, player_names):
    self.players = player_names
    self.hands = {name: [] for name in player_names}
    self.center_pile = []
    self.current_rank_index = 0
    self.last_play = None
    self.game_over = False

    deck = setup_deck()
    for _ in range(5): #  only dealing 5 cards per player for faster demo
      for name in self.players:
        self.hands[name].append(deck.pop())

  def get_target_rank(self):
    return RANKS[self.current_rank_index]

  def advance_rank(self):
    self.current_rank_index = (self.current_rank_index + 1) % len(RANKS)

  def print_state(self):
    print("\n" + "="*40)
    print(f"🎯 Target Rank: {self.get_target_rank()}")
    print(f"🃏 Center Pile: {len(self.center_pile)} cards")
    for p in self.players:
        print(f"👤 {p}: {len(self.hands[p])} cards")
    print("="*40 + "\n")

## crewAI Agent + task helpers

### crewAI agents:

crewAI `Agent` is defined by:
- `role` — label (like a job title)
- `goal` — what the agent is trying to achieve
- `backstory` — extra context/personality?
- `tools` — list of tool functions agent can call
- `llm` — the language model backend.
- `verbose` — if True, prints every thought/action step (great for debugging).

### crewAI tasks:

`Task` is a "single unit of work assigned to one agent":
- `description` — prompt / instructions for task.
- `expected_output` — example of good response.
- `agent` — which agent should do this task.

### messages/memory:

crewAI agents are stateless between `execute_task` calls (so each call is a new convoooooOOoooOo)

we gyatt to pass all the context needed (hand, target rank, chat log, etc.) directly in task description

this might look differently for a different application. for this game, the state changes every turn... but that might not be how ur application works

In [31]:
def make_agent(name: str, llm) -> Agent:
  return Agent(
    role=f"BS Card Game Player ({name})",
    goal="Empty your hand by playing cards — truthfully or by bluffing. Win the game.",
    backstory=(
      f"You are {name}, a cunning card shark. You know the rules of BS: "
      "each turn you must play 1-4 cards claiming they are the target rank. "
      "You can lie. Other players can call BS. If caught lying, you take the pile. "
      "If wrongly accused, the accuser takes the pile. Empty your hand to win."
    ),
    tools=PLAYER_TOOLS,
    llm=llm,
    verbose=True,   # --> outputs extra stuff in rich!
    allow_delegation=False,  # agents can't hand tasks off to each other
  )


def run_agent_task(agent: Agent, description: str, expected_output: str) -> str:
  task = Task(
      description=description,
      expected_output=expected_output,
      agent=agent,
  )
  result = agent.execute_task(task)
  # execute_task() returns a TaskOutput object
  # .raw gives the plain string.
  return result.raw if hasattr(result, 'raw') else str(result)
  # returns agents final output


def response_contains(result_str: str, keyword: str) -> bool:
  return keyword in result_str
  # checking agent output for keywords 😛

## main loop

maintain game information, check agent output for tool calls, execute them

In [32]:
def run_game():
  global game, current_player, all_agents, chat_log

  # ── LLM setup ─────────────────────────────────────────────────────────────
  llm = LLM(
    model="mistral/mistral-large-latest",
    api_key=os.environ.get("MISTRAL_API_KEY"),
  )

  # ── agent creation ────────────────────────────────────────────────────────
  names = ["Mehek", "Sania", "Kevin"]

  all_agents = {name: make_agent(name, llm) for name in names}

  # ── game loop ─────────────────────────────────────────────────────────────
  game = BSGame(names)
  chat_log = []
  turn_idx = 0

  while not game.game_over:
      game.print_state()
      active_name = names[turn_idx % len(names)]
      current_player = active_name
      target_rank = game.get_target_rank()

      # building chat context
      chat_context = ""
      if chat_log:
          recent = chat_log[-6:]  # last 6 messages to keep context short
          chat_context = "\n\nRecent table chat:\n" + "\n".join(recent)

      # ── PLAYER AGENT TURN ──────────────────────────────────────
      # inject the player's hand directly into the task descrip. (to keep it private)
      play_description = (
        f"It is your turn, {active_name}. "
        f"The target rank this round is: {target_rank}. "
        f"Your current hand is: {game.hands[active_name]}. "
        f"You MUST call the play_cards tool to play 1-4 cards from your hand. "
        f"You can play any cards but must claim they are {target_rank}s — you can bluff!"
        + chat_context
      )

      run_agent_task(
        all_agents[active_name],
        description=play_description,
        expected_output="Confirmation that cards were played using the play_cards tool.",
      )

      if game.last_play is None:
        # if gent didn't call play_cards--> skip turn (this is not supposed to happen, can invoke different behavior)
        turn_idx += 1
        continue

      # ── PLAYER REACTION ────────────────────────────────────────────
      bs_caller = None
      for other_name in names:
          if other_name == active_name:
            continue

          current_player = other_name  # update global so tools know which agent is calling

          react_description = (
            f"{active_name} just played {game.last_play['count']} card(s) "
            f"claiming they are {target_rank}s. "
            f"Your hand is: {game.hands[other_name]}. "
            f"Do you believe them, or do you think they're bluffing? "
            f"You MUST call either call_bs or pass_turn. "
            f"Think carefully — if you call BS and they were honest, YOU take the pile. "
            f"If you call BS and they lied, THEY take the pile."
            + chat_context
          )

          reaction_result = run_agent_task(
            all_agents[other_name],
            description=react_description,
            expected_output="Either 'BS_CALLED' if you called BS, or 'PASSED' if you let it go.",
          )

          if response_contains(reaction_result, "BS_CALLED"):
            bs_caller = other_name
            print(f"🚨 {bs_caller} CALLED BS ON {active_name}! 🚨") # this could also be directly printed in the tool?
            break

      # ── RESOLUTION OF TURN ──────────────────────────────────────────────
      if bs_caller:
        actual_cards = game.last_play["cards"]
        print(f"🔍 Revealing cards... {active_name} actually played: {actual_cards}")

        is_truth = all(c[:-1] == target_rank for c in actual_cards)

        if is_truth:
          print(f"✅ {active_name} told the TRUTH! {bs_caller} takes the pile.")
          game.hands[bs_caller].extend(game.center_pile)
        else:
          print(f"❌ {active_name} LIED! {active_name} takes the pile.")
          game.hands[active_name].extend(game.center_pile)

        game.center_pile = []

      game.last_play = None  # reset for next turn

      # ── 4. Check win condition ────────────────────────────────────────────
      for p in names:
        if len(game.hands[p]) == 0:
          print(f"\n🎉 {p} HAS WON THE GAME! 🎉")
          game.game_over = True
          break

      game.advance_rank()
      turn_idx += 1


run_game()

set up da card deck

🎯 Target Rank: A
🃏 Center Pile: 0 cards
👤 Mehek: 5 cards
👤 Sania: 5 cards
👤 Kevin: 5 cards



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: BS Card Game Player (Mehek)                                                                             │
│                                                                                                                 │
│  Task: It is your turn, Mehek. The target rank this round is: A. Your current hand is: ['2H', '2C', '9C',       │
│  '2D', '8C']. You MUST call the play_cards tool to play 1-4 cards from your hand. You can play any cards but    │
│  must claim they are As — you can bluff!                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🃏 Mehek played 3 card(s) claiming to be As.


╭──────────────────────────────────────── 🔧 Tool Execution Started (#35) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: play_cards                                                                                               │
│  Args: {'cards': ['2H', '2C', '2D']}                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool play_cards executed with result: Cards played successfully. You claimed they were As....


...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: BS Card Game Player (Mehek)                                                                             │
│                                                                                                                 │
│  Task: Kevin just played 2 card(s) claiming they are 6s. Your hand is: ['2H', '2C', '2D']. Do you believe       │
│  them, or do you think they're bluffing? You MUST call either call_bs or pass_turn. Think carefully — if you    │
│  call BS and they were honest, YOU take the pile. If you call BS and they lied, THEY take the pile.             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

An unknown error occurred. Please check the details below.
Error details: Event stack depth limit (100) exceeded. This usually indicates missing ending events.
An unknown error occurred. Please check the details below.
Error details: Event stack depth limit (100) exceeded. This usually indicates missing ending events.


[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_error' closed 'llm_call_started' (expected 
'agent_execution_started')

StackDepthExceededError: Event stack depth limit (100) exceeded. This usually indicates missing ending events.